# 00 Inspect Data

This notebook explores the raw Parquet files before designing the Bronze, Silver and Gold layers.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("supply-chain-data-platform")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 16:06:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pathlib import Path

data_dir = Path("../data/raw")
parquet_files = sorted(data_dir.glob("*.parquet"))

parquet_files

[PosixPath('../data/raw/customer_first.parquet'),
 PosixPath('../data/raw/customers_second.parquet'),
 PosixPath('../data/raw/orders_first.parquet'),
 PosixPath('../data/raw/orders_second.parquet'),
 PosixPath('../data/raw/products_first.parquet'),
 PosixPath('../data/raw/products_second.parquet'),
 PosixPath('../data/raw/regions.parquet')]

In [4]:
for file in parquet_files:
    print("=" * 80)
    print(f"File: {file.name}")
    
    df = spark.read.parquet(str(file))
    
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")
    print("Schema:")
    df.printSchema()
    
    print("Sample:")
    df.show(5, truncate=False)

File: customer_first.parquet
Rows: 1990
Columns: 6
Schema:
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)

Sample:
+-----------+----------+---------+------------------------+------------------+-----+
|customer_id|first_name|last_name|email                   |city              |state|
+-----------+----------+---------+------------------------+------------------+-----+
|C00001     |Emily     |Mooney   |rushjeff@ryan.org       |Johnsonmouth      |MS   |
|C00002     |Andrea    |Sellers  |mccoykiara@kelly.com    |Stephenfort       |WY   |
|C00003     |Craig     |Hayes    |rebeccamiller@yahoo.com |South Stephenshire|LA   |
|C00004     |Bryan     |Scott    |lawrence05@campbell.info|Chrisland         |ND   |
|C00005     |Sean      |Vasquez  |carrie45@yahoo.com      |East Dennistown   |RI   |
+----------

### Referential Integrity 

In [19]:
raw_tables = {
    file.stem: spark.read.parquet(str(file))
    for file in parquet_files
}

customers_first = raw_tables["customer_first"]
customers_second = raw_tables["customers_second"]
orders_first = raw_tables["orders_first"]
orders_second = raw_tables["orders_second"]
products_first = raw_tables["products_first"]
products_second = raw_tables["products_second"]
regions = raw_tables["regions"]

customers_all = customers_first.unionByName(customers_second)
orders_all = orders_first.unionByName(orders_second)
products_all = products_first.unionByName(products_second)

### Basic Structure 

In [20]:
overview = []

for name, df in raw_tables.items():
    overview.append({
        "table": name,
        "rows": df.count(),
        "columns": len(df.columns),
        "column_names": df.columns,
    })

import pandas as pd
pd.DataFrame(overview)

,table,rows,columns,column_names
0,customer_first,1990,6,"[customer_id, first_name, last_name, email, ci..."
1,customers_second,10,6,"[customer_id, first_name, last_name, email, ci..."
2,orders_first,9990,6,"[order_id, customer_id, product_id, order_date..."
3,orders_second,10,6,"[order_id, customer_id, product_id, order_date..."
4,products_first,490,5,"[product_id, product_name, category, brand, pr..."
5,products_second,10,5,"[product_id, product_name, category, brand, pr..."
6,regions,4,2,"[region_id, region]"


**Relations:**
customer <-> orders <-> products

### Schemas and Data Types

In [21]:
for name, df in raw_tables.items():
    print("=" * 80)
    print(name)
    df.printSchema()

customer_first
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)

customers_second
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)

orders_first
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)

orders_second
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullabl

### Primary Key Uniqueness

In [22]:
from pyspark.sql.functions import count, col

def duplicate_key_count(df, key_col):
    return (
        df
        .groupBy(key_col)
        .agg(count("*").alias("n"))
        .filter(col("n") > 1)
        .count()
    )

In [23]:
duplicate_checks = [
    ("customers_all", customers_all, "customer_id"),
    ("products_all", products_all, "product_id"),
    ("orders_all", orders_all, "order_id"),
    ("regions", regions, "region_id"),
]

for table_name, df, key_col in duplicate_checks:
    print(table_name, key_col, duplicate_key_count(df, key_col))

customers_all customer_id 0
products_all product_id 0
orders_all order_id 0
regions region_id 0


### Missing Values 

In [24]:
from pyspark.sql.functions import sum as spark_sum, when, col

def missing_value_summary(df):
    return df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

In [25]:
missing_value_summary(customers_all).show()
missing_value_summary(products_all).show()
missing_value_summary(orders_all).show()
missing_value_summary(regions).show()

+-----------+----------+---------+-----+----+-----+
|customer_id|first_name|last_name|email|city|state|
+-----------+----------+---------+-----+----+-----+
|          0|         0|        0|    0|   0|    0|
+-----------+----------+---------+-----+----+-----+

+----------+------------+--------+-----+-----+
|product_id|product_name|category|brand|price|
+----------+------------+--------+-----+-----+
|         0|           0|       0|    0|    0|
+----------+------------+--------+-----+-----+

+--------+-----------+----------+----------+--------+------------+
|order_id|customer_id|product_id|order_date|quantity|total_amount|
+--------+-----------+----------+----------+--------+------------+
|       0|          0|         0|         0|       0|           0|
+--------+-----------+----------+----------+--------+------------+

+---------+------+
|region_id|region|
+---------+------+
|        0|     0|
+---------+------+



### Referential Integrity 

In [26]:
def missing_reference_count(fact_df, dimension_df, key_col):
    return (
        fact_df
        .join(dimension_df, on=key_col, how="left_anti")
        .count()
    )

In [27]:
print(
    "orders_all.customer_id missing in customers_all:",
    missing_reference_count(orders_all, customers_all, "customer_id")
)

print(
    "orders_all.product_id missing in products_all:",
    missing_reference_count(orders_all, products_all, "product_id")
)

orders_all.customer_id missing in customers_all: 0
orders_all.product_id missing in products_all: 0


### Batch Behavior: First vs. Second 

In [29]:
batch_overlap_checks = [
    ("orders", orders_first, orders_second, "order_id"),
    ("customers", customers_first, customers_second, "customer_id"),
    ("products", products_first, products_second, "product_id"),
]

for entity, first_df, second_df, key_col in batch_overlap_checks:
    overlap = second_df.join(first_df, on=key_col, how="inner").count()
    print(entity, "overlap:", overlap)

orders overlap: 0
customers overlap: 0
products overlap: 0


### Value Ranges and Obvious Anomalies 

In [30]:
from pyspark.sql.functions import min as spark_min, max as spark_max, avg

orders_all.select(
    spark_min("order_date").alias("min_order_date"),
    spark_max("order_date").alias("max_order_date"),
    spark_min("quantity").alias("min_quantity"),
    spark_max("quantity").alias("max_quantity"),
    avg("quantity").alias("avg_quantity"),
    spark_min("total_amount").alias("min_total_amount"),
    spark_max("total_amount").alias("max_total_amount"),
    avg("total_amount").alias("avg_total_amount"),
).show()

+--------------+--------------+------------+------------+------------+----------------+----------------+------------------+
|min_order_date|max_order_date|min_quantity|max_quantity|avg_quantity|min_total_amount|max_total_amount|  avg_total_amount|
+--------------+--------------+------------+------------+------------+----------------+----------------+------------------+
|    2023-01-01|    2024-12-31|           1|           5|       3.017|           12.35|          9952.9|3136.0156080000143|
+--------------+--------------+------------+------------+------------+----------------+----------------+------------------+



### Candidate Business Questions 

In [37]:
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round

In [38]:
orders_all.groupBy("product_id").agg(
    spark_sum("quantity").alias("total_quantity"),
    spark_round(spark_sum("total_amount"), 2).alias("total_amount")
).orderBy(col("total_quantity").desc()).show(10)

+----------+--------------+------------+
|product_id|total_quantity|total_amount|
+----------+--------------+------------+
|     P0266|           113|    36653.81|
|     P0076|           102|    87998.46|
|     P0496|            99|    13624.38|
|     P0296|            99|   172126.35|
|     P0267|            97|   171964.51|
|     P0271|            97|   132185.78|
|     P0247|            97|     80034.7|
|     P0262|            97|    85859.55|
|     P0112|            96|   151654.08|
|     P0045|            95|    59119.45|
+----------+--------------+------------+
only showing top 10 rows


In [39]:
orders_products = orders_all.join(products_all, on="product_id", how="left")

orders_products.groupBy("category").agg(
    spark_sum("quantity").alias("total_quantity"),
    spark_round(spark_sum("total_amount"), 2).alias("total_amount")
).orderBy(col("total_quantity").desc()).show(10)

+-----------+--------------+------------+
|   category|total_quantity|total_amount|
+-----------+--------------+------------+
|     Beauty|          6020|  6413514.12|
|Electronics|          5755|  6153060.29|
|       Toys|          5345|  5340086.54|
|     Sports|          4759|  5117373.47|
|       Home|          4183|  4003424.91|
|   Clothing|          4108|  4332696.75|
+-----------+--------------+------------+



## Findings

The dataset contains orders, customers, products and regions.

The main relationships are:

- `orders.customer_id` → `customers.customer_id`
- `orders.product_id` → `products.product_id`

The `first` and `second` files appear to represent separate ingestion batches. The second batch contains new records rather than updates, since no overlapping primary keys were found between first and second files.

The `regions` table currently has no direct foreign key relationship to the other tables.

## Bronze design implication

The Bronze layer should not clean or join the data. It should store raw records as close to the source as possible, while adding metadata:

- source file name
- ingestion timestamp
- batch label, for example `first` or `second`